**하나의 제품 정보로 인스타그램 · 보도자료 · 사내공지 3개 채널의 홍보 문구를 생성하고, 채널별 토큰 사용량과 비용을 표로 정리하라.**

* 제품 정보: 무선 이어폰 SOSIN Buds — 노이즈 캔슬링, 배터리 30시간, 무게 4.5g, 가격 89,000원

* 채널마다 instructions로 역할(페르소나), 문체, 문장 수를 다르게 지정한다.

* 채널마다 max_output_tokens를 다르게 설정하고, 응답의 status로 답변이 잘렸는지 확인한다.

* usage로 입력 · 출력 토큰을 구하고, gpt-6-luna 단가(입력 $0.10, 출력 $0.50 / 1M)로 채널별 비용(USD, 원)과 총비용을 계산한다.

* 결과를 pandas DataFrame으로 출력한다.



In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

import os
MY_API_KEY=os.environ['OPENAI_API_KEY']

client=OpenAI()
MODEL='gpt-6-luna'

In [9]:
PRICE = {
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "output": 50.00},
    "gpt-6-sol":   {"input": 2.00,  "cached": 0.20, "output": 10.00},
    "gpt-6-luna":  {"input": 0.10,  "cached": 0.01, "output": 0.50},
}
USD_KRW = 1350  # 환율 가정

def cost_usd(resp, model=MODEL):
    p = PRICE[model]
    u = resp.usage
    cached = 0
    if getattr(u, "input_tokens_details", None) is not None:
        cached = u.input_tokens_details.cached_tokens or 0
    return (
        (u.input_tokens - cached) * p["input"]
        + cached * p["cached"]
        + u.output_tokens * p["output"]
    ) / 1_000_000

In [30]:
# 인스타그램
response = client.responses.create(
    model=MODEL,
    instructions=(
        '당신은 제품의 판촉을 담당하는 마케팅 부서 직원입니다.'
        'input은 제품의 정보를 담고 있음'
        '제품 정보를 모두 포함하여'
        '인스타그램 분위기에 맞게 홍보 문구를 작성'
        '2줄 이내로 응답할 것'
    ),
    input='무선 이어폰 SOSIN Buds — 노이즈 캔슬링, 배터리 30시간, 무게 4.5g, 가격 89,000원',
)
print(response.output_text)

it1=response.usage.input_tokens
ot1=response.usage.output_tokens
print(f'input tokens: {it1} / output tokens: {ot1}')
c1=cost_usd(response)
print(f"{c1:.6f} USD  (약 {c1 * USD_KRW:.2f}원)")

🎧 SOSIN Buds로 소음은 차단하고, 몰입은 UP! 노이즈 캔슬링에 4.5g 초경량, 배터리 최대 30시간까지✨  
지금 만나보세요, 89,000원 💫
input tokens: 99 / output tokens: 101
0.000060 USD  (약 0.08원)


In [29]:
# 공식 보도자료
response=client.responses.create(
    model=MODEL,
    instructions=(
        '당신은 제품의 판촉을 담당하는 마케팅 부서 직원입니다.'
        'input은 제품의 정보를 담고 있음'
        '제품 정보를 모두 포함하여'
        '언론에 제공할 보도자료 문구를 작성'
        '평어체, 문어체로 작성'
        '8줄 내외로 응답할 것'
    ),
    input='무선 이어폰 SOSIN Buds — 노이즈 캔슬링, 배터리 30시간, 무게 4.5g, 가격 89,000원',
    # max_output_tokens=90
)
print(response.output_text)

it2=response.usage.input_tokens
ot2=response.usage.output_tokens
print(f'input tokens: {it2} / output tokens: {ot2}')
c2=cost_usd(response)
print(f"{c2:.6f} USD  (약 {c2 * USD_KRW:.2f}원)")

**SOSIN Buds, 편안한 착용감과 강력한 노이즈 캔슬링으로 몰입감 높여**

무선 이어폰 SOSIN Buds가 노이즈 캔슬링 기능으로 더욱 집중도 높은 청취 환경을 제공한다.  
주변 소음을 줄여 음악 감상과 통화에 몰입할 수 있도록 설계했다.  
이어버드 무게는 4.5g으로 가벼워 장시간 착용에도 부담을 덜어준다.  
배터리는 최대 30시간 사용할 수 있어 외출과 여행에도 편리하다.  
가벼운 착용감과 긴 사용 시간을 바탕으로 일상 속 활용도를 높였다.  
SOSIN Buds의 판매 가격은 89,000원이다.  
노이즈 캔슬링과 휴대성을 갖춘 무선 이어폰을 찾는 소비자에게 적합한 제품이다.
input tokens: 107 / output tokens: 263
0.000142 USD  (약 0.19원)


In [28]:
# 사내 공지
response=client.responses.create(
    model=MODEL,
    instructions=(
        '당신은 제품의 판촉을 담당하는 마케팅 부서 직원입니다.'
        'input은 제품의 정보를 담고 있음'
        '제품 정보를 모두 포함하여'
        '자사 직원에게 안내 메일을 작성하세요.'
        '경어체, 구어체로 작성'
        '8줄 내외로 응답할 것'
    ),
    input='무선 이어폰 SOSIN Buds — 노이즈 캔슬링, 배터리 30시간, 무게 4.5g, 가격 89,000원',
)
print(response.output_text)

it3=response.usage.input_tokens
ot3=response.usage.output_tokens
print(f'input tokens: {it3} / output tokens: {ot3}')
c3=cost_usd(response)
print(f"{c3:.6f} USD  (약 {c3 * USD_KRW:.2f}원)")

제목: 무선 이어폰 SOSIN Buds 판촉 안내  
안녕하세요, 임직원 여러분.  
이번에 새롭게 선보이는 무선 이어폰 **SOSIN Buds**를 안내드립니다.  
주변 소음을 줄여주는 **노이즈 캔슬링** 기능을 갖췄습니다.  
배터리는 최대 **30시간** 사용할 수 있어 편리합니다.  
이어폰 한쪽 무게는 **4.5g**으로 가볍게 착용하실 수 있습니다.  
가격은 **89,000원**입니다.  
제품 홍보와 안내에 많은 관심 부탁드립니다. 감사합니다.
input tokens: 106 / output tokens: 182
0.000102 USD  (약 0.14원)


max_output_tokens: 90 미만에선 출력조차 되지 않고, 90부터 반 줄 정도 출력되기 시작

In [33]:
import pandas as pd

# 채널별 토큰 사용량과 비용
df = pd.DataFrame({
    '채널': ['인스타그램', '보도자료', '사내공지'],
    '입력 토큰': [it1, it2, it3],
    '출력 토큰': [ot1, ot2, ot3],
    '비용(USD)': [c1, c2, c3],
})
df['비용(원)'] = df['비용(USD)'] * USD_KRW

# 합계 행 추가
df.loc[len(df)] = ['합계', df['입력 토큰'].sum(), df['출력 토큰'].sum(), df['비용(USD)'].sum(), df['비용(원)'].sum()]

df.style.format({'비용(USD)': '{:.6f}', '비용(원)': '{:.2f}'})

,채널,입력 토큰,출력 토큰,비용(USD),비용(원)
0,인스타그램,99,101,0.000060,0.08
1,보도자료,107,263,0.000142,0.19
2,사내공지,106,182,0.000102,0.14
3,합계,312,546,0.000304,0.41


---

**콘솔에서 대화하는 영어 회화 튜터를 만들어라.**

* instructions: 초보자용 쉬운 영어로 2문장 이내로 답하고 후속 질문 1개를 한다. 사용자 문장에 문법 오류가 있으면 교정: <수정 문장> 한 줄을 덧붙인다.

* 답변은 스트리밍(stream=True)으로 글자가 흘러나오듯 출력한다.

* previous_response_id로 이전 대화를 이어간다.

* 턴마다 누적 입력 · 출력 토큰을 출력하고, quit / exit / 종료를 입력하면 대화를 끝낸다.

* 테스트: 3턴 이상 대화하면서, 앞에서 말한 내용(예: 내 취미)을 튜터가 기억하는지 확인한다.

In [34]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

import os
MY_API_KEY=os.environ['OPENAI_API_KEY']

client=OpenAI()
MODEL='gpt-6-luna'

In [ ]:
# 영어 회화 튜터
INSTRUCTIONS = (
    '당신은 초보자를 위한 영어 회화 튜터입니다.'
    '초보자용 쉬운 영어로 2문장 이내로 답할 것'
    '답변 뒤에 후속 질문 1개를 영어로 할 것'
    '사용자 문장에 문법 오류가 있으면 마지막에 "교정: <수정 문장>" 한 줄을 덧붙일 것'
    '문법 오류가 없으면 교정 줄은 생략'
)

prev_id = None  # 이전 응답 id
itk = 0    # 누적 입력 토큰
otk = 0   # 누적 출력 토큰

while True:
    user_text = input('You: ').strip()
    if user_text.lower() in ('quit', 'exit', '종료'):
        print('대화를 종료합니다.')
        break
    if not user_text:
        continue
    print(f'You: {user_text}')

    # instructions는 다음 턴으로 이어지지 않으므로 매 턴 전달
    stream = client.responses.create(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=user_text,
        previous_response_id=prev_id,
        stream=True,
    )

    print('Tutor: ', end='', flush=True)
    for event in stream:
        if event.type == 'response.output_text.delta':
            print(event.delta, end='', flush=True)
        elif event.type == 'response.completed':
            prev_id = event.response.id
            itk += event.response.usage.input_tokens
            otk += event.response.usage.output_tokens
    print()
    print(f'[누적] input tokens: {itk} / output tokens: {otk}\n')

You: Hello
Tutor: Hello! How are you today?
[누적] input tokens: 102 / output tokens: 61

You: Fine, how about you?
Tutor: I’m doing well, thank you! What are you doing today?
[누적] input tokens: 277 / output tokens: 131

You: just relaxing in home
Tutor: That sounds nice! What do you like to do while relaxing?  
교정: I’m just relaxing at home.
[누적] input tokens: 532 / output tokens: 296

You: playing PC games, playing with my cat, watching Netflixes, very much like this.
Tutor: That sounds fun! What PC games do you like to play?  
교정: I like playing PC games, playing with my cat, and watching Netflix. I really like doing these things.
[누적] input tokens: 976 / output tokens: 508

대화를 종료합니다.


---

**여러 개의 회의록에서 요약과 액션 아이템(담당자, 할 일, 기한)을 추출해 CSV로 저장하라.**

* Pydantic으로 ActionItem(owner, task, due)와 MeetingResult(summary, action_items) 스키마를 정의하고 responses.parse로 추출한다.

* 기한이 없으면 "미정", 액션 아이템이 없는 회의(잡담)는 빈 리스트로 처리한다.

* RateLimitError, APIConnectionError가 발생하면 지수 백오프로 최대 3회 재시도하고, 끝내 실패하면 건너뛴다.

* 추출 결과를 회의명과 함께 DataFrame으로 모아 action_items.csv로 저장한다.

* 테스트 데이터: 액션 아이템이 3건인 회의, 2건인 회의, 0건인 잡담 대화 각 1개

    * 주간회의_0922: "김대리: 신규 랜딩페이지 시안은 목요일까지 제가 공유할게요. 박과장: 광고 예산안은 다음 주 월요일까지 정리하겠습니다. 이팀장: 고객 인터뷰 일정은 추후 정하죠.”

    * 개발회의_0923: "최개발: 로그인 버그는 오늘 중으로 수정합니다. 정QA: 수정되면 회귀 테스트를 수요일까지 진행할게요.”

    * 잡담: "오늘 점심 뭐 먹을까요? 비가 와서 국물이 좋겠네요.”



In [37]:
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

import os
MY_API_KEY=os.environ['OPENAI_API_KEY']

client=OpenAI()
MODEL='gpt-6-luna'

In [36]:
from pydantic import BaseModel
from openai import RateLimitError, APIConnectionError
import pandas as pd

# 추출 스키마
class ActionItem(BaseModel):
    owner: str  # 담당자
    task: str   # 할 일
    due: str    # 기한, 없으면 "미정"

class MeetingResult(BaseModel):
    summary: str
    action_items: list[ActionItem]

In [ ]:
# 테스트 데이터
meetings = {
    '주간회의_0922': '김대리: 신규 랜딩페이지 시안은 목요일까지 제가 공유할게요. 박과장: 광고 예산안은 다음 주 월요일까지 정리하겠습니다. 이팀장: 고객 인터뷰 일정은 추후 정하죠.',
    '개발회의_0923': '최개발: 로그인 버그는 오늘 중으로 수정합니다. 정QA: 수정되면 회귀 테스트를 수요일까지 진행할게요.',
    '잡담': '오늘 점심 뭐 먹을까요? 비가 와서 국물이 좋겠네요.',
}

INSTRUCTIONS = (
    '당신은 회의록을 정리하는 비서입니다.'
    'input은 회의록 원문입니다.'
    'summary에는 회의 내용을 한 문장으로 요약할 것'
    'action_items에는 담당자(owner), 할 일(task), 기한(due)을 추출'
    '기한이 언급되지 않으면 due는 "미정"으로 작성'
    '할 일이 없는 대화(잡담)는 action_items를 빈 리스트로 반환'
)

# SDK 내장 재시도: RateLimitError, APIConnectionError 발생 시 최대 3회 재시도
retry_client = client.with_options(max_retries=3)

def extract(text):
    try:
        response = retry_client.responses.parse(
            model=MODEL,
            instructions=INSTRUCTIONS,
            input=text,
            text_format=MeetingResult,
        )
        return response.output_parsed
    except (RateLimitError, APIConnectionError) as e:
        # 3회 재시도 후에도 실패하면 None 반환
        print(f'재시도 실패: {type(e).__name__}')
        return None

In [39]:
# 회의별 추출 후 DataFrame으로 수집
rows = []
for name, text in meetings.items():
    result = extract(text)
    if result is None:
        print(f'[{name}] 건너뜀')
        continue
    print(f'[{name}] {result.summary} / 액션 아이템 {len(result.action_items)}건')

    # 액션 아이템이 없는 회의도 요약은 남김
    if not result.action_items:
        rows.append({'회의명': name, '요약': result.summary, '담당자': None, '할 일': None, '기한': None})
    for item in result.action_items:
        rows.append({'회의명': name, '요약': result.summary, '담당자': item.owner, '할 일': item.task, '기한': item.due})

df = pd.DataFrame(rows)
df.to_csv('action_items.csv', index=False, encoding='utf-8-sig')
df

[주간회의_0922] 신규 랜딩페이지 시안과 광고 예산안을 각각 목요일과 다음 주 월요일까지 준비하고, 고객 인터뷰 일정은 추후 정하기로 했습니다. / 액션 아이템 3건
[개발회의_0923] 로그인 버그를 수정한 뒤 회귀 테스트를 진행하기로 했습니다. / 액션 아이템 2건
[잡담] 비가 오는 날씨를 고려해 점심으로 국물 요리가 좋겠다는 의견을 나눴습니다. / 액션 아이템 0건


,회의명,요약,담당자,할 일,기한
0,주간회의_0922,"신규 랜딩페이지 시안과 광고 예산안을 각각 목요일과 다음 주 월요일까지 준비하고, ...",김대리,신규 랜딩페이지 시안 공유,목요일
1,주간회의_0922,"신규 랜딩페이지 시안과 광고 예산안을 각각 목요일과 다음 주 월요일까지 준비하고, ...",박과장,광고 예산안 정리,다음 주 월요일
2,주간회의_0922,"신규 랜딩페이지 시안과 광고 예산안을 각각 목요일과 다음 주 월요일까지 준비하고, ...",이팀장,고객 인터뷰 일정 정하기,미정
3,개발회의_0923,로그인 버그를 수정한 뒤 회귀 테스트를 진행하기로 했습니다.,최개발,로그인 버그 수정,오늘
4,개발회의_0923,로그인 버그를 수정한 뒤 회귀 테스트를 진행하기로 했습니다.,정QA,수정 완료 후 회귀 테스트 진행,수요일까지
5,잡담,비가 오는 날씨를 고려해 점심으로 국물 요리가 좋겠다는 의견을 나눴습니다.,NaN,NaN,NaN
